In [ ]:
import ast
import pandas as pd

precursor_df = pd.read_csv("precursor_protein_level.tsv", sep="\t")
reasons = [
    "IDENTIFIED",
    "non_standard_modification",
    "Others",
    "miss_3_cleavages",
    "HLA1",
    "HLA2",
    "lost_3_aa_Nterm",
    "multiple_modifications",
    "peptide_length_>_40",
    "regular_tryptic",
    "C_without_+57_modification",
]

records = []
for reason in reasons:
    proteins = set()
    peptides = set()
    for _, row in precursor_df.iterrows():
        if row["n_noncontained_le9"] <=1:
            continue
        scans = []
        if pd.notna(row["evidence_peptides_scans"]):
            try:
                scans = ast.literal_eval(row["evidence_peptides_scans"])
            except Exception:
                continue
        for scan in scans:
            if len(scan) >= 3 and scan[2] == reason:
                proteins.add(row["precursor_protein"])
                if len(scan) >= 4 and scan[3]:
                    peptides.add(scan[0])
    records.append(
        {
            "reason": reason,
            "protein_count": len(proteins),
            "proteins": sorted(proteins),
            "evidence_peptide_count": len(peptides),
            "evidence_peptides": peptides,
        }
    )

reason_level_df = pd.DataFrame(records)
reason_level_df.to_csv("reason_level.tsv", sep="\t", index=False)

: 